# **Statistical Significance Tests**

## **Objetivo del notebook**

Este notebook tiene como propósito **validar estadísticamente** que el modelo principal (Stacking Ensemble) supera de manera significativa a una batería de modelos benchmark en la tarea de clasificación de crédito (`credit_score`).

Para ello se implementa un **Bootstrap Paired Significance Test**, que es el método más adecuado cuando:
- Las predicciones de todos los modelos se obtienen sobre el **mismo conjunto de prueba** (muestras pareadas).
- No se desea asumir distribución normal de los errores.
- Se trabaja con métricas compuestas como F1-macro.

### **Modelos comparados**
| Modelo | Tipo |
|---|---|
| **Original model** | Stacking Ensemble (modelo propio) |
| Naive Bayes | Baseline probabilístico |
| KNN | Baseline por distancia |
| Logistic Regression | Baseline lineal |
| Random Forest | Ensemble de árboles |
| SVM | Máquina de vectores de soporte |
| Ridge | Clasificador lineal regularizado |
| XGBoost | Gradient boosting |
| Decision Tree | Árbol individual |

### **Métrica de evaluación**
Se usa **F1-macro**, que promedia el F1 de cada clase sin ponderar por frecuencia. Es la métrica correcta dado el desbalance de clases presente en este dataset.

In [1]:
import numpy as np
from Lava_transformer import LavaTransformer
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

## **Carga de datos**

Se carga el dataset ya preprocesado (`data_limpia.pkl`) y se define la variable objetivo `credit_score`.

La variable objetivo se convierte a valores numéricos mediante un **mapeo ordinal** consistente con el orden conceptual del riesgo crediticio:
- `Good → 0`
- `Standard → 1`
- `Poor → 2`

> **Nota sobre el desbalance:** El dataset presenta un desbalance moderado: Standard (~53%), Poor (~29%), Good (~18%). Por eso se usa F1-macro como métrica principal, y se aplica `stratify=y` en el split para preservar estas proporciones en train y test.

In [2]:
from collections import Counter
df = pd.read_pickle("data_limpia.pkl")

target_col = "credit_score"
y_raw = df[target_col].copy()
X = df.drop(columns=[target_col]).copy() 

print("Shape df:", df.shape)
print("Clases originales:")
print(y_raw.value_counts())

# Codificación de la variable objetivo
class_order = ["Good", "Standard", "Poor"]
mapping = {cls: i for i, cls in enumerate(class_order)}
inverse_mapping = {i: cls for cls, i in mapping.items()}

y = y_raw.map(mapping) # Convertir a valores numéricos

print("\nMapping:", mapping)
print("Distribución y:", Counter(y))

Shape df: (100000, 29)
Clases originales:
credit_score
Standard    53174
Poor        28998
Good        17828
Name: count, dtype: int64

Mapping: {'Good': 0, 'Standard': 1, 'Poor': 2}
Distribución y: Counter({1: 53174, 2: 28998, 0: 17828})


## **Partición de datos** 

Se realiza una partición **70% entrenamiento / 30% prueba** con `random_state=42` para reproducibilidad.

El parámetro `stratify=y` garantiza que la distribución de clases se preserve en ambos splits, lo cual es crítico cuando hay desbalance. Con 100,000 registros, el conjunto de prueba cuenta con **30,000 muestras**, un tamaño más que suficiente para que el Bootstrap Paired Test produzca estimaciones estables y confiables.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Valores únicos y_test:", np.unique(y_test))


X_train: (70000, 28)
X_test : (30000, 28)
Valores únicos y_test: [0 1 2]


## **Modelos a comparar**

Se definen las rutas de los archivos `.pkl` de cada modelo. Los benchmarks fueron reentrenados bajo las mismas condiciones que el modelo original para garantizar una comparación justa.

- El **Original model** es un Stacking Ensemble personalizado con arquitectura propia (ver función `predict_stacking`).
- Los modelos en `models_reentrenados/` son los mejores hiperparámetros encontrados para cada algoritmo, reentrenados sobre el mismo `X_train`.

In [4]:
all_model_paths = {
    "Original model": r"models/stacking_model_final.pkl",
    "Naive Bayes": r"models_reentrenados/best_nb_model_optimized_reentrenado.pkl",
    "Knn": r"models_reentrenados/best_knn_model_optimized_reentrenado.pkl",
    "logistic regression": r"models_reentrenados/best_lr_model_optimized_reentrenado.pkl",
    "Random Forest": r"models_reentrenados/best_rf_model_optimized_reentrenado.pkl",
    "SVM": r"models_reentrenados/best_svc_model_optimized_reentrenado.pkl",
    "Ridge": r"models_reentrenados/best_ridge_model_optimized_reentrenado.pkl",
    "xgb": r"models_reentrenados/best_xgboost_model_optimized_reentrenado.pkl",
    "Decision tree": r"models_reentrenados/best_dt_model_optimized_reentrenado.pkl",
}

## **Función `predict`**

### **Arquitectura del Stacking Ensemble**

La función `predict_stacking` implementa manualmente el flujo de inferencia del modelo original, que sigue la siguiente arquitectura de dos niveles:

```
X_test
  │
  ▼
base_pipeline (preprocesamiento)
  │
  ├──► KNN  ──► probs_knn  ─┐
  ├──► RF   ──► probs_rf   ─┼──► blend ponderado
  └──► XGB  ──► probs_xgb  ─┘        │
                                      ▼
                           [confidence, entropy]
                                      │
                              meta_features (hstack)
                                      │
                                      ▼
                                 meta_model
                                      │
                           calibración por alphas
                                      │
                                 predicción final
```

**Componentes clave:**
- `blend`: promedio ponderado de las probabilidades de los modelos base.
- `confidence`: probabilidad máxima del blend (qué tan seguro está el ensemble).
- `entropy`: incertidumbre del blend — alta entropía indica una predicción poco confiable.
- `alphas`: factores de calibración por clase que ajustan las probabilidades del meta-modelo antes del `argmax`.

In [5]:
# funcion para el modelo de stacking

def predict_stacking(stacking_obj, X):
    base_pipe = stacking_obj["base_pipeline"]
    base_models = stacking_obj["base_models"]
    meta_model = stacking_obj["meta_model"]
    weights = stacking_obj["weights"]
    alphas = np.asarray(stacking_obj["alphas"], dtype=float)
    class_order_model = stacking_obj["class_order"]

    X_proc = base_pipe.transform(X)

    probs = {k: m.predict_proba(X_proc) for k, m in base_models.items()}
    blend = sum(weights[k] * probs[k] for k in weights)

    confidence = blend.max(axis=1).reshape(-1, 1)
    entropy = -(blend * np.log(blend + 1e-15)).sum(axis=1).reshape(-1, 1)

    meta_X = np.hstack([
        probs["knn"],
        probs["rf"],
        probs["xgb"],
        blend,
        confidence,
        entropy
    ])

    proba = meta_model.predict_proba(meta_X)
    proba = proba / alphas
    idx_pred = proba.argmax(axis=1)

    y_pred_labels = np.array([class_order_model[i] for i in idx_pred])
    return y_pred_labels

### **Generación de predicciones para todos los modelos**

El bucle de carga maneja tres casos de formato de serialización:
1. **Original model**: usa `predict_stacking` con el flujo propio.
2. **Dict con clave `best_model`**: algunos benchmarks fueron guardados como diccionario, se extrae el modelo internamente.
3. **Pipeline o modelo sklearn directo**: el caso más común para los benchmarks.

En todos los casos, las predicciones se normalizan al espacio numérico `{0, 1, 2}` para poder comparar con `y_test`.

In [6]:
# función para modelos benchmark

all_preds = {}
results = {}

for name, path in all_model_paths.items():
    obj = joblib.load(path)

    # Original model
    if name == "Original model":
        y_pred_labels = predict_stacking(obj, X_test)
        y_pred = np.array([mapping[label] for label in y_pred_labels])

    # Si algún benchmark quedó guardado como dict
    elif isinstance(obj, dict):
        model = obj["best_model"]
        raw_pred = model.predict(X_test)

        if isinstance(raw_pred[0], str):
            y_pred = np.array([mapping[label] for label in raw_pred])
        else:
            y_pred = np.asarray(raw_pred)

    # Pipeline/modelo sklearn-imblearn normal
    else:
        raw_pred = obj.predict(X_test)

        if isinstance(raw_pred[0], str):
            y_pred = np.array([mapping[label] for label in raw_pred])
        else:
            y_pred = np.asarray(raw_pred)

    all_preds[name] = y_pred
    results[name] = f1_score(y_test, y_pred, average="macro")

    print(f"{name}: {results[name]:.6f}")

Original model: 0.798543
Naive Bayes: 0.630356
Knn: 0.773537
logistic regression: 0.640412
Random Forest: 0.766519
SVM: 0.637492
Ridge: 0.633108
xgb: 0.768637
Decision tree: 0.684667


## **Chequeo de alineación**

Verificación de integridad antes de proceder con los tests estadísticos. Se comprueba que:
- `y_test` contiene exactamente las clases `[0, 1, 2]`.
- Las predicciones de **todos** los modelos también cubren las tres clases.

Esto es importante porque si algún modelo nunca predice una clase, el F1-macro se calcula sobre menos clases y los valores no serían comparables.

In [7]:
print("Valores únicos y_test:", np.unique(y_test))

for name, preds in all_preds.items():
    print(name, "-> únicos:", np.unique(preds), "| F1:", f1_score(y_test, preds, average="macro"))

Valores únicos y_test: [0 1 2]
Original model -> únicos: [0 1 2] | F1: 0.7985427811377854
Naive Bayes -> únicos: [0 1 2] | F1: 0.6303555996509552
Knn -> únicos: [0 1 2] | F1: 0.7735366253414216
logistic regression -> únicos: [0 1 2] | F1: 0.6404118694447435
Random Forest -> únicos: [0 1 2] | F1: 0.7665186653460863
SVM -> únicos: [0 1 2] | F1: 0.6374920103123826
Ridge -> únicos: [0 1 2] | F1: 0.6331075784093695
xgb -> únicos: [0 1 2] | F1: 0.768636700958453
Decision tree -> únicos: [0 1 2] | F1: 0.6846671338474355


## **Tabla de resultados**

Resumen del F1-macro obtenido por cada modelo sobre el conjunto de prueba (30,000 muestras). Los resultados se presentan en orden descendente para facilitar la comparación visual.

A primera vista, el modelo original lidera con un F1-macro de **0.7985**, seguido de cerca por los tres mejores benchmarks (KNN, XGBoost, Random Forest). Sin embargo, esta diferencia observada en un único conjunto de prueba **no demuestra por sí sola que sea estadísticamente significativa** — para eso se realiza el Bootstrap Test en la siguiente sección.

In [8]:
df_results = pd.DataFrame(
    [{"Model": model, "F1_macro": score} for model, score in results.items()]
).sort_values("F1_macro", ascending=False).reset_index(drop=True)

pd.options.display.float_format = '{:.6f}'.format

print("Tabla ordenada:")
display(df_results)

Tabla ordenada:


,Model,F1_macro
0,Original model,0.798543
1,Knn,0.773537
2,xgb,0.768637
3,Random Forest,0.766519
4,Decision tree,0.684667
5,logistic regression,0.640412
6,SVM,0.637492
7,Ridge,0.633108
8,Naive Bayes,0.630356


## **Función Bootstrap**

### **¿Qué es el Bootstrap Paired Significance Test?**

Es un método de remuestreo no paramétrico para estimar si la diferencia de rendimiento entre dos modelos es **estadísticamente significativa** o podría explicarse por varianza aleatoria del conjunto de prueba.

**Procedimiento paso a paso:**
1. Se tienen `n` muestras en el conjunto de prueba.
2. En cada iteración bootstrap, se seleccionan `n` índices **con reemplazo** (algunas muestras se repiten, otras se omiten).
3. Se calcula el F1-macro del modelo original y del benchmark sobre ese subconjunto remuestreado.
4. Se registra la diferencia: `diff = F1_original - F1_benchmark`.
5. Tras `n_bootstrap=2000` iteraciones, se obtiene la distribución empírica de las diferencias.

**Métricas de salida:**
- `mean_diff`: diferencia promedio de F1 (qué tanto supera el modelo original al benchmark en promedio).
- `ci_low / ci_high`: intervalo de confianza al 95% (percentiles 2.5 y 97.5 de la distribución de diferencias).
- `p_value = P(diff ≤ 0)`: fracción de iteraciones en las que el benchmark igualó o superó al modelo original. Es un **test unilateral** bajo la hipótesis de que el modelo original es mejor.

**Criterio de significancia:** `p_value < 0.05` → la superioridad del modelo original es estadísticamente significativa al 95% de confianza.

In [9]:
import numpy as np
from sklearn.metrics import f1_score

def bootstrap_f1_test(y_true, y_pred_1, y_pred_2,
                      n_bootstrap=2000,
                      average="macro",
                      random_state=42):

    y_true = np.array(y_true)
    y_pred_1 = np.array(y_pred_1)
    y_pred_2 = np.array(y_pred_2)

    rng = np.random.RandomState(random_state)
    n = len(y_true)

    diffs = []

    for _ in range(n_bootstrap):
        idx = rng.choice(np.arange(n), size=n, replace=True)

        f1_1 = f1_score(y_true[idx], y_pred_1[idx], average=average)
        f1_2 = f1_score(y_true[idx], y_pred_2[idx], average=average)

        diffs.append(f1_1 - f1_2)

    diffs = np.array(diffs)

    mean_diff = diffs.mean()
    ci_low = np.percentile(diffs, 2.5)
    ci_high = np.percentile(diffs, 97.5)

    # p-value (test unilateral): fracción de iteraciones en que el benchmark igualó o superó al original
    p_value = np.mean(diffs <= 0)

    return mean_diff, ci_low, ci_high, p_value

## **Significance test: Original vs benchmarks**

Se ejecuta `bootstrap_f1_test` comparando el **Original model** contra cada uno de los 8 benchmarks.

Configuración del test:
- `n_bootstrap = 2000`: número de remuestreos. Con 30,000 muestras de prueba, este valor produce estimaciones estables.
- `average = "macro"`: consistente con la métrica de evaluación principal.
- `random_state = 42`: garantiza reproducibilidad exacta de los resultados.

Los resultados se ordenan por `mean_diff` descendente para visualizar primero los benchmarks más fácilmente superados.

In [10]:
comparison_results = {}

reference_model = "Original model"

for name, y_pred in all_preds.items():
    if name == reference_model:
        continue

    mean_diff, ci_low, ci_high, p_value = bootstrap_f1_test(
        y_true=y_test,
        y_pred_1=all_preds[reference_model],
        y_pred_2=y_pred,
        n_bootstrap=2000,
        average="macro",
        random_state=42
    )

    comparison_results[name] = {
        "mean_diff": mean_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "significant": p_value < 0.05
    }

df_significance = pd.DataFrame(comparison_results).T.sort_values(
    "mean_diff", ascending=False
)

display(df_significance)

,mean_diff,ci_low,ci_high,p_value,significant
Naive Bayes,0.168233,0.163007,0.173375,0.000000,True
Ridge,0.165447,0.160150,0.170528,0.000000,True
SVM,0.161045,0.155530,0.166222,0.000000,True
logistic regression,0.158125,0.152988,0.163043,0.000000,True
Decision tree,0.113877,0.108888,0.118936,0.000000,True
Random Forest,0.032048,0.027849,0.036016,0.000000,True
xgb,0.029921,0.026120,0.033880,0.000000,True
Knn,0.025055,0.020958,0.029070,0.000000,True


## **Interpretación**



La conclusión se determina a partir del **intervalo de confianza al 95%** de la diferencia de F1:
- Si `ci_low > 0`: todo el intervalo es positivo → el modelo original **siempre** fue mejor en las 2000 muestras bootstrap → **Original supera al benchmark de forma significativa**.
- Si `ci_high < 0`: todo el intervalo es negativo → el benchmark es superior.
- Si el intervalo cruza el 0: **no hay diferencia estadísticamente significativa**.

En este caso, los resultados son contundentes: en ninguna de las 2000 iteraciones bootstrap ningún benchmark igualó o superó al modelo original (`p_value = 0.0` en todos los casos). Esto significa que con más de **2,000 muestras de evidencia**, la superioridad del Stacking Ensemble es **robusta** y no producto de varianza aleatoria.

**Análisis por grupos de competidores:**

| Grupo | Modelos | mean_diff | Interpretación |
|---|---|---|---|
| Baselines débiles | Naive Bayes, Ridge, SVM, LR | ~0.16–0.17 | El original los supera ampliamente (~16 pts F1) |
| Árbol individual | Decision Tree | ~0.11 | Mejora sustancial (~11 pts F1) |
| Ensembles fuertes | Random Forest, XGBoost, KNN | ~0.025–0.032 | Ventaja más ajustada pero igual significativa (~2.5–3 pts F1) |

In [11]:
for model, row in df_significance.iterrows():
    if row["ci_low"] > 0:
        conclusion = "Original > benchmark (significativo)"
    elif row["ci_high"] < 0:
        conclusion = "Benchmark > Original (significativo)"
    else:
        conclusion = "No diferencia significativa"

    print(f"{model}: {conclusion}")

Naive Bayes: Original > benchmark (significativo)
Ridge: Original > benchmark (significativo)
SVM: Original > benchmark (significativo)
logistic regression: Original > benchmark (significativo)
Decision tree: Original > benchmark (significativo)
Random Forest: Original > benchmark (significativo)
xgb: Original > benchmark (significativo)
Knn: Original > benchmark (significativo)


## **Conclusión**
El **Bootstrap Paired Significance Test** con 2,000 iteraciones sobre 30,000 muestras de prueba demuestra que el **Stacking Ensemble (Original model)** supera estadísticamente a **todos los benchmarks** evaluados, con un nivel de confianza del 95%.

Puntos clave:
- **p_value = 0.0** en todos los casos: en ninguna de las 2,000 muestras bootstrap un benchmark igualó o superó al modelo original.
- **Los intervalos de confianza no cruzan el cero** en ningún caso: la superioridad es robusta y no se explica por varianza del conjunto de prueba.
- La ventaja más pequeña (vs. KNN: +0.025 F1-macro) sigue siendo estadísticamente significativa, lo que indica que incluso frente al benchmark más competitivo, el Stacking Ensemble aporta valor real.
- La métrica F1-macro, elegida por el desbalance de clases, garantiza que los resultados reflejan el rendimiento equilibrado en las tres categorías de crédito: Good, Standard y Poor.
